## 1. Imports

In [2]:
import pandas as pd

df_clean = pd.read_csv(
    r"C:\Users\carine\mars26_bds_immo\data\raw\df_clean.csv"
)

df_ML = df_clean.copy()

df_ML.info()

<class 'pandas.DataFrame'>
RangeIndex: 5818581 entries, 0 to 5818580
Data columns (total 19 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   annee                      int64  
 1   nature_mutation            str    
 2   valeur_fonciere            float64
 3   surface_reelle_bati        float64
 4   nombre_pieces_principales  float64
 5   surface_terrain            float64
 6   longitude                  float64
 7   latitude                   float64
 8   surface_log                float64
 9   type_local_code            int64  
 10  departement_code           int64  
 11  t                          int64  
 12  mois_sin                   float64
 13  mois_cos                   float64
 14  trimestre                  int64  
 15  is_vefa                    int64  
 16  surface_par_piece          float64
 17  anomalie_structure         int64  
 18  surface_terrain_log        float64
dtypes: float64(11), int64(7), str(1)
memory usage: 843.5 

In [3]:
import numpy as np
df_ML["prix_log"] = np.log1p(df_ML["valeur_fonciere"])

In [4]:
cols_to_drop = [
    "valeur_fonciere",     # fuite de données
    "t",                   # variable artificielle
    "type_local_code",     # encodage arbitraire
    "departement_code",    # encodage trompeur
    "anomalie_structure"   # variable trop rare
]

df_ML = df_ML.drop(columns=cols_to_drop, errors="ignore")

In [5]:
df_ML.corr(numeric_only=True)["prix_log"].sort_values(ascending=False)

prix_log                     1.000000
surface_reelle_bati          0.402703
surface_log                  0.391852
nombre_pieces_principales    0.335508
surface_terrain_log          0.242724
surface_par_piece            0.141597
is_vefa                      0.059226
surface_terrain              0.057025
longitude                    0.013727
annee                        0.013550
trimestre                    0.006474
latitude                    -0.004782
mois_cos                    -0.011752
mois_sin                    -0.011777
Name: prix_log, dtype: float64

## 2. Encoding

In [6]:
target = "prix_log"

X = df_ML.drop(columns=[target])
y = df_ML[target]

## 3. Split train/test

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(4654864, 14) (1163717, 14)
(4654864,) (1163717,)


In [9]:
X_train.isnull().sum().sum()

np.int64(3795904)

In [10]:
X_train.isnull().sum().sort_values(ascending=False)

surface_terrain              1814252
surface_terrain_log          1814252
latitude                       83700
longitude                      83700
nature_mutation                    0
annee                              0
surface_reelle_bati                0
nombre_pieces_principales          0
mois_sin                           0
surface_log                        0
mois_cos                           0
trimestre                          0
is_vefa                            0
surface_par_piece                  0
dtype: int64

In [11]:
df_ML[df_ML["surface_terrain"].isnull()].head()

,annee,nature_mutation,surface_reelle_bati,nombre_pieces_principales,surface_terrain,longitude,latitude,surface_log,mois_sin,mois_cos,trimestre,is_vefa,surface_par_piece,surface_terrain_log,prix_log
5,2020,Vente en l'état futur d'achèvement,61.0,3.0,NaN,4.838408,46.303181,4.127134,-0.5,-0.866025,3,1,20.333333,NaN,12.100551
6,2020,Vente en l'état futur d'achèvement,118.0,4.0,NaN,5.217651,46.208039,4.779123,-0.5,-0.866025,3,1,29.500000,NaN,12.793862
7,2020,Vente,62.0,3.0,NaN,5.219443,46.198796,4.143135,-0.5,-0.866025,3,0,20.666667,NaN,11.820418
10,2020,Vente en l'état futur d'achèvement,53.0,2.0,NaN,5.126002,46.337021,3.988984,-0.5,-0.866025,3,1,26.500000,NaN,11.728045
11,2020,Vente,111.0,2.0,NaN,5.224621,46.208263,4.718499,-0.5,-0.866025,3,0,55.500000,NaN,12.506181


In [12]:
df_ML["has_terrain"] = df_ML["surface_terrain"].notna().astype(int)

In [13]:
df_ML["surface_terrain"] = df_ML["surface_terrain"].fillna(0)
df_ML["surface_terrain_log"] = np.log1p(df_ML["surface_terrain"])

In [14]:
df_ML["has_terrain"] = df_ML["surface_terrain"].notna().astype(int)

df_ML["surface_terrain"] = df_ML["surface_terrain"].fillna(0)

df_ML["surface_terrain_log"] = np.log1p(df_ML["surface_terrain"])

In [15]:
df_ML["has_terrain"] = (df_ML["surface_terrain"].notna()).astype(int)

## Features SUR X_train ET X_test

In [16]:
# Indicateur terrain
X_train["has_terrain"] = X_train["surface_terrain"].notna().astype(int)
X_test["has_terrain"] = X_test["surface_terrain"].notna().astype(int)

In [17]:
#gestion des NaN terrain
X_train["surface_terrain"] = X_train["surface_terrain"].fillna(0)
X_test["surface_terrain"] = X_test["surface_terrain"].fillna(0)

In [18]:
# transformation LOG
import numpy as np

X_train["surface_terrain_log"] = np.log1p(X_train["surface_terrain"])
X_test["surface_terrain_log"] = np.log1p(X_test["surface_terrain"])

In [19]:
# fEATUres VEFA X TERRAIN
X_train["vefa_no_terrain"] = X_train["is_vefa"] * (X_train["has_terrain"] == 0).astype(int)
X_test["vefa_no_terrain"] = X_test["is_vefa"] * (X_test["has_terrain"] == 0).astype(int)

In [20]:
X_train.isnull().sum().sum()
X_test.isnull().sum().sum()

np.int64(41274)

In [21]:
X_train.isnull().sum().sort_values(ascending=False).head(10)

latitude                     83700
longitude                    83700
annee                            0
nature_mutation                  0
nombre_pieces_principales        0
surface_reelle_bati              0
surface_terrain                  0
surface_log                      0
mois_sin                         0
mois_cos                         0
dtype: int64

In [22]:
from sklearn.impute import SimpleImputer

In [23]:
imputer = SimpleImputer(strategy="median")

In [24]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object"]).columns

C:\Users\carine\AppData\Local\Temp\ipykernel_196\2590777664.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(include=["object"]).columns


In [25]:
X_train = pd.get_dummies(X_train, columns=["nature_mutation"], drop_first=True)
X_test = pd.get_dummies(X_test, columns=["nature_mutation"], drop_first=True)

In [26]:
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

In [27]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

## 4. Models XGBoost

In [29]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ----- ---------------------------------- 13.6/101.7 MB 61.4 MB/s eta 0:00:02
   ----------- ---------------------------- 29.9/101.7 MB 68.3 MB/s eta 0:00:02
   ----------------- ---------------------- 44.3/101.7 MB 69.5 MB/s eta 0:00:01
   ----------------------- ---------------- 59.2/101.7 MB 69.5 MB/s eta 0:00:01
   ----------------------------- ---------- 75.0/101.7 MB 70.2 MB/s eta 0:00:01
   ----------------------------------- ---- 91.2/101.7 MB 71.0 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 71.5 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 71.5 MB/s eta 0:00:01
   --------------------------------------  101.4/101.7 MB 71.5 MB/s eta 0:00:01
   ---------------------------------------- 101.7/101.7 MB 49.6 MB/s  0:00:02
Note: you may need to restart the kernel to use updated pack


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train_imputed, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

## 5. Evaluation

In [32]:
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np



In [33]:
preds_xgb = xgb.predict(X_test_imputed)

rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
r2_xgb = r2_score(y_test, preds_xgb)

print("RMSE XGB:", rmse_xgb)
print("R2 XGB:", r2_xgb)

RMSE XGB: 0.49148491907935726
R2 XGB: 0.6003884342145728


## 6. Relance modèle amélioré

In [34]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=800,        # ↑ très important
    learning_rate=0.03,      # ↓ plus stable
    max_depth=10,            # ↑ capacité
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)

xgb.fit(X_train_imputed, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.8
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

## 7. Evaluation

In [35]:
preds_xgb = xgb.predict(X_test_imputed)

rmse_xgb = np.sqrt(mean_squared_error(y_test, preds_xgb))
r2_xgb = r2_score(y_test, preds_xgb)

print("RMSE XGB:", rmse_xgb)
print("R2 XGB:", r2_xgb)

RMSE XGB: 0.46364481757688714
R2 XGB: 0.6443781177352247


## eSSAIS AUTRE TUNING 

In [37]:
xgb = XGBRegressor(
    n_estimators=1200,
    learning_rate=0.02,
    max_depth=12,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    random_state=42
)